# Finding the data: CESM2 LENS on AWS

**Does the autumn ENSO state predict a wet Front Range winter?** This cookbook answers that one classification question on NSF NCAR's
CESM2 Large Ensemble, following the analysis spine of NSF Unidata's *Supervised Machine Learning Readiness*. This first chapter frames the
question and finds the data.

*Code: Apache-2.0 · Content: CC BY 4.0 · Data: NCAR CESM2 Large Ensemble Cloud-Optimized Dataset (Grover & Banihirwe 2021, https://doi.org/10.26024/y48t-q717) and Rodgers et al. 2021 (https://doi.org/10.5194/esd-12-1393-2021), UCAR Data Terms of Use; Community Earth System Model v2 Large Ensemble (CESM2 LENS) was accessed on 2026-10-20 from https://registry.opendata.aws/ncar-cesm2-lens.*

*Drafted with AI assistance (Anthropic's Claude) from cited sources, under the author's direction, and reviewed by the author in several full read-throughs.*

*Text and code were checked against those sources and the notebooks' own outputs by automated tests and independent AI reviewers; no human subject-matter expert has reviewed it yet.*


## Overview

A classifier will read one number — a member's autumn Niño 3.4 state — and say whether its coming Front Range winter will be wet. Before
any model, this chapter:

1. states the question as a decision;
2. answers Unidata's three framing questions;
3. locates the data in the `intake-esm` catalogue and records it in a Metadata Document;
4. loads the committed table and asserts the parameters it was derived with.

The spine is Unidata's (Apache-2.0), except that the split is **by ensemble member** (chapter 3). The README has a glossary. Shared code lives in `lensml.py` beside the chapters (parameters, transforms, ladder, scoring, paired test, register); `show()` prints a
function's source where a chapter introduces it.


## Prerequisites

| Concepts | Importance | Notes |
|---|---|---|
| [Xarray tutorial](https://tutorial.xarray.dev/) | Necessary | a Zarr store, a box selection, `groupby` |
| [Intake-ESM](https://intake-esm.readthedocs.io/) | Helpful | one catalogue search |
| scikit-learn | Not required | every call is introduced where it is used |

- **Time to learn**: about 20 minutes. Every cell runs in seconds from the committed cache; the catalogue cells need a network route to AWS.
- **System requirements**: the `environment.yml` at the repository root; no account, no local data beyond the committed cache.


## Imports

The second cell prints the package versions this run resolved; one that disagrees with `environment.yml` means the wrong kernel.


In [ ]:
import importlib.metadata
import re
import sys
from datetime import datetime

import intake
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from lensml import (
    FRONT_RANGE,
    NINO34,
    PARAMS_RECORD,
    SEASON,
    credit_own,
    derive_member_winters,
    load_cache,
    show,
)

In [ ]:
now = datetime.now().astimezone().isoformat(timespec="seconds")
print(f"run started {now}   kernel python {sys.version.split()[0]}")
PACKAGES = [
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "matplotlib",
    "xarray",
    "zarr",
    "s3fs",
    "cftime",
    "intake",
    "intake-esm",
]
for pkg in PACKAGES:
    try:
        print(f"{pkg:13s} {importlib.metadata.version(pkg)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg:13s} NOT INSTALLED: re-create the environment")

## The question, as a decision

Every parameter in `lensml.py` is a design decision with its reason; changing one is a trial you record. The seed is declared once and passed down — never `np.random.seed()` (scikit-learn's *Common pitfalls*, §12.3.3).

The person who acts on the answer is the forecaster drafting a local winter-outlook briefing, such as NWS Boulder's (September 2020) or
NWS Pueblo's (2025). The official wet, dry or "equal chances" lean is the Climate Prediction Center's; the briefing relays it and says what
the ENSO state suggests locally (Pueblo's 2025 briefing does both). A "wet" call right more often than the base rate, without missing most
wet winters, would let the briefing lean wet where the official outlook says "equal chances". The figure draws the question's two boxes from the parameters (no data, no coastlines).


In [ ]:
def span(box):
    """A box as degrees with hemispheres, from its slices on the 0-360 grid."""
    y0, y1 = box["lat"].start, box["lat"].stop
    x0, x1 = box["lon"].start, box["lon"].stop

    def lat(y):
        return f"{abs(y):g}°{'S' if y < 0 else 'N'}"

    def lon(x):
        return f"{x:g}°E" if x <= 180 else f"{360 - x:g}°W"

    return f"{lat(y0)}–{lat(y1)}, {lon(x0)}–{lon(x1)}"


fig, ax = plt.subplots(figsize=(9, 3.8))
ax.set(xlim=(165, 275), ylim=(-24, 56), ylabel="latitude")
ax.set_title(
    "The question's two boxes: the autumn (SON) Niño 3.4 state as the input, "
    f"{SEASON} Front Range precipitation as the target",
    fontsize=9,
)
xt = list(range(170, 276, 15))
ax.set_xticks(xt)
ax.set_xticklabels(
    [f"{x}°E" if x <= 180 else f"{360 - x}°W" for x in xt], fontsize=8
)
yt = list(range(-20, 61, 20))
ax.set_yticks(yt)
ax.set_yticklabels(
    [f"{abs(y)}°{'S' if y < 0 else 'N'}" for y in yt], fontsize=8
)
ax.axhline(0, lw=0.8, c="#6A7077", ls=":")
ax.grid(True, lw=0.5)
boxes = [
    (
        "NINO34",
        NINO34,
        "input: SON mean of TS (Rodgers et al. 2021 box)",
        (215, -13),
        "top",
    ),
    (
        "FRONT_RANGE",
        FRONT_RANGE,
        f"target: {SEASON} mean of PRECC + PRECL (Front Range box around Denver: "
        "west of the Divide to the plains)",
        (236, 50),
        "bottom",
    ),
]
for name, box, note, at, va in boxes:
    x0, x1 = box["lon"].start, box["lon"].stop
    y0, y1 = box["lat"].start, box["lat"].stop
    ax.add_patch(
        plt.Rectangle(
            (x0, y0),
            x1 - x0,
            y1 - y0,
            facecolor="#C9A23A",
            edgecolor="#8F7022",
            lw=1.2,
            zorder=3,
        )
    )
    ax.annotate(
        f"{name}  {span(box)}\n{note}",
        xy=((x0 + x1) / 2, y0 if va == "top" else y1),
        xytext=at,
        fontsize=7.5,
        ha="center",
        va=va,
        arrowprops=dict(arrowstyle="-", color="#6A7077", lw=0.8),
    )
ax.annotate(
    "",
    xy=(FRONT_RANGE["lon"].start, FRONT_RANGE["lat"].start),
    xytext=(NINO34["lon"].stop, NINO34["lat"].stop),
    arrowprops=dict(
        arrowstyle="->",
        color="#8F7022",
        lw=1.6,
        connectionstyle="arc3,rad=-0.25",
    ),
)
ax.text(
    243,
    22,
    f"SON of year y − 1\n→ {SEASON} winter y",
    fontsize=8,
    ha="right",
    va="center",
    color="#8F7022",
)
credit_own(fig, "a diagram of the parameter boxes; no CESM2 LENS data drawn")
plt.tight_layout()

```{admonition} Try it: the expected sign
:class: tip
Before any data cell runs, write down the sign you expect: does a warm autumn Pacific make a wetter or a drier Front Range winter? NWS
Boulder places El Niño's wet signal east of the Divide mainly in late winter and spring, so `SEASON` changes how much signal there is to
learn; DJF is the committed path, FMA costs a rebuild (below).
```


## Framing: Unidata's three questions

Unidata opens every analysis with three questions, guarding against a target that answers no one's decision.

```{admonition} Try it: frame it yourself first
:class: tip
Before opening the worked answers, write your own: three simple solutions a model must beat, whether the requirements can be met, and
the question as one decision.
```

<details>
<summary><b>The worked answers</b></summary>

**Does a simpler solution exist?** Three, and a model is measured against them, not assumed to beat them.

| Simple solution | What it predicts | What it sets |
|---|---|---|
| always-dry | "dry" every winter | a bar on accuracy only (wet precision and recall are 0) |
| always-wet | "wet" every winter | recall 1 at precision = the base rate: the bar every precision is read against |
| Niño-sign rule | "wet" when the autumn Niño 3.4 anomaly is positive | the physical prior (NWS Boulder east of the Divide: El Niño wetter, La Niña drier); no bar |

**Can machine learning requirements be met?** On the data side, yes: the catalogue holds monthly PRECC, PRECL and TS for every historical
member in both forcing variants. On the physics side the honest prior is a *small* effect: Stagge et al. (2023) find the ENSO
winter-precipitation signal over the Intermountain West is a north–south dipole, "the northern pole … north of 40° N" and "the southern
pole … south of 35° N", the Front Range in the "transition zone (32–40° N)" between them, east of the "steep eastern slope" where the northern signal
"dissipates abruptly"; the northern pole's sign is reversed
(La Niña wetter). The signal is measured here, not assumed.

**Which scientific question should be answered?** *"Given the autumn Niño 3.4 state in a CESM2 LENS member, will that member's coming
Front Range winter be wet — above the one pooled threshold that marks the upper third of the training members' detrended winters?"*

</details>


## The catalogue on AWS

The catalogue forces two facts on the design: there is **no monthly PRECT**, so the target is `PRECC + PRECL`; and there is **no variable named SST** (and no ocean `TEMP`
for the historical-smbb half), so the Niño 3.4 index is built from atmospheric `TS` over open ocean.

Offline, each cell below prints `OFFLINE` and the chapter runs on from the cache. The second computes what a box subset reads.


In [ ]:
CATALOG_URL = "https://ncar-cesm2-lens.s3-us-west-2.amazonaws.com/catalogs/aws-cesm2-le.json"
try:
    cat = intake.open_esm_datastore(CATALOG_URL)
    subset = cat.search(
        component="atm",
        frequency="monthly",
        experiment="historical",
        variable=["PRECC", "PRECL", "TS"],
    )
    print(
        f"catalogue last_updated: {cat.esmcat.last_updated}   "
        f"rows in catalogue: {len(cat.df)}   rows selected: {len(subset.df)}"
    )
    cols = [
        "variable",
        "forcing_variant",
        "units",
        "start_time",
        "end_time",
        "path",
    ]
    with pd.option_context(
        "display.max_colwidth", None
    ):  # the six paths in full
        display(subset.df[cols])
except Exception as err:  # no network, a DNS failure, S3 unreachable
    cat = subset = None
    print(
        f"OFFLINE: the catalogue at {CATALOG_URL} could not be read "
        f"({type(err).__name__}). The Metadata Document's catalogue and store "
        "rows are *not printed this run*; every later cell runs from the cache."
    )

In [ ]:
try:
    probe = xr.open_zarr(
        "s3://ncar-cesm2-lens/atm/monthly/cesm2LE-historical-cmip6-TS.zarr",
        consolidated=True,
        storage_options={"anon": True},
    )
except (
    Exception
) as err:  # offline: the structure rows are not printed this run
    probe = None
    print(
        f"OFFLINE: the TS store could not be opened ({type(err).__name__}); "
        "the dims, calendar and chunking rows are *not printed this run*."
    )
if probe is not None:
    print("dims      :", dict(probe.sizes))
    print("coords    :", list(probe.coords))
    print(
        "members   :",
        probe.member_id.values[:3],
        "…",
        "calendar:",
        type(probe.time.values[0]).__name__,
    )
    ch = tuple(int(c) for c in probe["TS"].encoding["chunks"])
    per_chunk = (
        int(np.prod(ch)) * probe["TS"].dtype.itemsize
    )  # bytes, uncompressed
    n_time_chunks = -(
        -probe.sizes["time"] // ch[1]
    )  # chunks along time per member
    per_member_var = n_time_chunks * per_chunk
    n_members = (
        2 * probe.member_id.size
    )  # two halves, each the size of this store
    print(
        f"chunking  : {ch} {probe['TS'].dtype} → {per_chunk / 1e6:.0f} MB per chunk "
        f"uncompressed; one member-variable ({probe.sizes['time']} months) = "
        f"{n_time_chunks} chunks × {per_chunk / 1e6:.0f} MB ≈ {per_member_var / 1e6:.0f} "
        f"MB read for any box subset; the derivation ({n_members} members × 3 "
        f"variables) ≈ {n_members * 3 * per_member_var / 1e9:.0f} GB"
    )

## The Metadata Document

Unidata's Metadata Document has four parts — general, structure, quality, provenance — each row filled from the evidence it names.

| Part | Field | Value | Evidence |
|---|---|---|---|
| General | Dataset | CESM2 Large Ensemble on AWS Open Data, historical experiment, 1850–2014 | AWS Open Data registry page |
| General | Licence, citation | UCAR Data Terms of Use (CC BY 4.0): cite the dataset DOI and the paper (Resources) | registry page |
| Structure | Stores used | PRECC, PRECL, TS × forcing variants cmip6, smbb | the catalogue cell's paths |
| Structure | Grid, calendar, units | ≈1°, monthly, noleap; PRECC/PRECL m/s → mm/day × 8.64e7; TS in K | the `dims` and `members` lines |
| Structure | Members | two forcing halves of equal size; 20 micro-members per AMOC start date | CESM2 LE project page; Rodgers et al. 2021 §2.2 |
| Structure | One row = | one member-winter: raw SON Niño 3.4 and season-mean Front Range PRECT | `derive_member_winters` |
| Quality | Known gap | ocean TEMP absent for historical-smbb → TS used for both halves | catalogue |
| Quality | The two halves | smbb = smoothed biomass-burning forcing 1990–2020 + two minor bug fixes: check base rates per half (chapter 2) | CESM2 LE project page |
| Quality | Window | from `WINDOW[0]`: members "should not be considered … independent" soon after 1850; chapter 3 measures what is left | Rodgers et al. 2021 §2.2 |
| Provenance | Catalogue | `aws-cesm2-le.json`; documentation DOI 10.26024/y48t-q717 | the `last_updated` line; registry |
| Provenance | Access | anonymous S3, us-west-2; `ACCESS_DATE` for the committed cache, asserted against the cache header | registry; the cache cell |

```{admonition} Try it: check the document
:class: tip
Mark every row the printed output can confirm *checked*, and any it cannot this run (offline, say) *not printed this run*. For each
*Quality* row, write one sentence on the bias it could introduce if ignored.
```


## The member-winter table

Two box means for every member read the volume the `chunking` line computes, so the derivation runs **once** and writes `data/member_winters.csv`, which every chapter reads. Its reader **asserts** that the header records the current
`SEASON`, `WINDOW`, `FRONT_RANGE`, `NINO34` and `AUTUMN_MONTHS`, and an `access=` date equal to `ACCESS_DATE`: a decision the cache
silently ignored would be a fit that cannot be rerun. One exception: the committed header predates the `AUTUMN_MONTHS` field, so the reader
assumes SON (the only autumn the code then had) and prints a line saying so. A rebuild records the field.


In [ ]:
show(derive_member_winters, load_cache)

**How a rebuild works.** `REBUILD = True` below re-derives the table from S3 with the parameters in `lensml.py`. The full ensemble is a
background job (33 minutes on a fast link, 2026-10-20; an hour or more on home broadband); fewer members is a quick sample. A rebuild is a new access: it stamps its own date into the header, and the chapters then ask you to set `ACCESS_DATE` to match.


In [ ]:
REBUILD = False  # True re-derives data/member_winters.csv from S3
N_MEMBERS_PER_HALF = (
    50  # used only when REBUILD = True; 50 = the full ensemble
)

table, cache_header, access_date = load_cache(REBUILD, N_MEMBERS_PER_HALF)
print(cache_header)
print(
    "cache header matches lensml.py:", PARAMS_RECORD, f"· access={access_date}"
)
n_header = 2 * int(
    re.search(r"n_members_per_half=(\d+)", cache_header).group(1)
)
n_store = (
    2 * probe.member_id.size if probe is not None else None
)  # unknown offline
if n_store:
    kind = (
        "A SAMPLE"
        if table.member_id.nunique() < n_store
        else "the full ensemble"
    )
    of_store = f"of {n_store} in the two stores ({kind})"
else:
    of_store = (
        f"(offline: the header says it was derived from {n_header} members)"
    )
print(
    f"{len(table)} member-winters · {table.member_id.nunique()} members {of_store} "
    f"· winters {table.winter.min()}–{table.winter.max()}"
)
table.head()

## Summary

The question is a forecaster's autumn decision, measured against three simple rules. The data are recorded in a Metadata Document whose
rows point at printed evidence; the member-winter table is derived once and asserted against the parameters on every read.

### What's next?

[Chapter 2](02-target-and-forced-response.ipynb) explores the target and the input, removes the forced response on the training members
only, and defines "wet".

## Resources and references

| Resource | Where it bears | Licence |
|---|---|---|
| Grover, M., & Banihirwe, A. (2021). *NCAR CESM2 Large Ensemble Cloud-Optimized Dataset*. NSF NCAR CISL. https://doi.org/10.26024/y48t-q717 | the dataset (required citation) | UCAR Data Terms of Use |
| Rodgers, K. B. et al. (2021). Ubiquity of human-induced changes in climate variability. *Earth System Dynamics* 12, 1393–1411. https://doi.org/10.5194/esd-12-1393-2021 | the ensemble paper; the Window row; the Niño 3.4 box | CC BY 4.0 |
| CESM2 LENS on the Registry of Open Data on AWS. https://registry.opendata.aws/ncar-cesm2-lens/ | access | UCAR Data Terms of Use |
| CESM2 Large Ensemble project page. https://www.cesm.ucar.edu/community-projects/lens2 | members, micro-members, smbb | UCAR website terms of use |
| NSF Unidata, *Supervised Machine Learning Readiness* (NSF award #2319979). https://github.com/Unidata/supervised-ml-readiness | the spine; the three framing questions | Apache-2.0 |
| Stagge, J. H. et al. (2023). *Nature Water* 1, 1016–1026. https://doi.org/10.1038/s44221-023-00163-9 | the ENSO dipole | CC BY 4.0 |
